
### Features prévues

**AgeGroup** — tranches d'âge jeune / adulte / senior
→ on a vu en EDA que l'âge influence le churn

**HasBalance** — 0 si Balance=0, 1 si Balance>0
→ un client sans solde est moins engagé dans la banque

**BalanceRatio** — Balance / EstimatedSalary
→ un solde élevé par rapport au salaire signifie un client très impliqué

**IsMultiProduct** — 1 si NumOfProducts >= 2, 0 sinon
→ les clients avec 3-4 produits churnen massivement (confirmé Kruskal-Wallis)

**EngagementScore** — IsActiveMember + (NumOfProducts / 4)
→ score d'engagement global du client

**SeniorInactive** — 1 si Age > 40 ET IsActiveMember = 0
→ client senior inactif = profil à risque élevé de churn

In [1]:
# Objectif : Créer de nouvelles features à partir des variables existantes
# Input    : data/processed/train.csv + test.csv
# Output   : data/processed/train_fe.csv + test_fe.csv
# Auteur   : Romaric TCHOFFO
# Date     : 2026

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv("../data/processed/train.csv")
test  = pd.read_csv("../data/processed/test.csv")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"\nColonnes : {list(train.columns)}")

Train : (12000, 14)
Test  : (3000, 14)

Colonnes : ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'SatisfactionScore', 'NumComplaints', 'Geography_Germany', 'Geography_Spain', 'Exited']


In [2]:
# On applique les mêmes transformations sur train ET test
# pour éviter tout data leakage

for df in [train, test]:

    # AgeGroup — tranches d'âge basées sur la logique métier bancaire
    # on a vu en EDA que les churners ont en moyenne 2.3 ans de plus
    df["AgeGroup"] = pd.cut(df["Age"],
                            bins=[-np.inf, 0, 1, 2, np.inf],
                            labels=[0, 1, 2, 3]).astype(int)

    # HasBalance — client avec ou sans solde
    # un client avec Balance=0 est moins engagé financièrement
    df["HasBalance"] = (df["Balance"] > 0).astype(int)

    # BalanceRatio — rapport solde / salaire
    # capture l'implication financière relative du client
    # on clip à 10 pour éviter les valeurs infinies si salaire=0
    df["BalanceRatio"] = (df["Balance"] / (df["EstimatedSalary"] + 1)).clip(upper=10)

    # IsMultiProduct — client avec 2+ produits
    # confirmé par Kruskal-Wallis : NumOfProducts influence fortement le churn
    df["IsMultiProduct"] = (df["NumOfProducts"] >= 2).astype(int)

    # EngagementScore — score d'engagement global
    # combine activité et nombre de produits
    
    df["EngagementScore"] = df["IsActiveMember"] + (df["NumOfProducts"] / 4)

    # SeniorInactive — client senior ET inactif
    # profil à risque élevé identifié en EDA
    # Age est standardisé → 0 correspond à la médiane (40 ans)
    df["SeniorInactive"] = ((df["Age"] > 0) & (df["IsActiveMember"] == 0)).astype(int)


In [3]:

print("Features créées avec succès !")
print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"\nNouvelles colonnes : {['AgeGroup','HasBalance','BalanceRatio','IsMultiProduct','EngagementScore','SeniorInactive']}")

# Vérification des corrélations des nouvelles features avec Exited
nouvelles_features = ["AgeGroup", "HasBalance", "BalanceRatio",
                      "IsMultiProduct", "EngagementScore", "SeniorInactive"]

print("\nCorrélation avec Exited :")
print(train[nouvelles_features + ["Exited"]].corr()["Exited"].drop("Exited").round(4))

Features créées avec succès !
Train : (12000, 20)
Test  : (3000, 20)

Nouvelles colonnes : ['AgeGroup', 'HasBalance', 'BalanceRatio', 'IsMultiProduct', 'EngagementScore', 'SeniorInactive']

Corrélation avec Exited :
AgeGroup           0.0577
HasBalance         0.0024
BalanceRatio       0.0092
IsMultiProduct     0.0533
EngagementScore    0.0408
SeniorInactive     0.0303
Name: Exited, dtype: float64


In [4]:
import os

os.makedirs("../data/processed", exist_ok=True)

train.to_csv("../data/processed/train_fe.csv", index=False)
test.to_csv("../data/processed/test_fe.csv",   index=False)

print(f"train_fe.csv sauvegardé : {train.shape}")
print(f"test_fe.csv  sauvegardé : {test.shape}")

train_fe.csv sauvegardé : (12000, 20)
test_fe.csv  sauvegardé : (3000, 20)


# Conclusions Générales — Feature Engineering ChurnGuard

## Features créées

**AgeGroup** — tranches d'âge encodées en 0/1/2/3
→ corrélation avec Exited : 0.0577 — meilleure nouvelle feature

**HasBalance** — 1 si le client a un solde, 0 sinon
→ corrélation avec Exited : 0.0024

**BalanceRatio** — Balance / EstimatedSalary clippé à 10
→ corrélation avec Exited : 0.0092

**IsMultiProduct** — 1 si NumOfProducts >= 2
→ corrélation avec Exited : 0.0533 — confirmé par Kruskal-Wallis

**EngagementScore** — IsActiveMember + (NumOfProducts / 4)
→ corrélation avec Exited : 0.0408

**SeniorInactive** — 1 si Age > 0 ET IsActiveMember = 0
→ corrélation avec Exited : 0.0303

## Dataset final

train_fe.csv : 12 000 lignes × 20 colonnes
test_fe.csv  : 3 000 lignes × 20 colonnes

## Prochaine étape

05_model_training.ipynb : entraînement des modèles Logistic Regression,
Decision Tree et Random Forest avec MLflow tracking.